In [ ]:
!apt-get update -qq
!apt-get install -y zstd

W: https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2404/x86_64/InRelease: Key is stored in legacy trusted.gpg keyring (/etc/apt/trusted.gpg), see the DEPRECATION section in apt-key(8) for details.
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 85 not upgraded.
Need to get 644 kB of archives.
After this operation, 1,845 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu noble-updates/main amd64 zstd amd64 1.5.5+dfsg2-2build1.1 [644 kB]
Fetched 644 kB in 2s (341 kB/s)
Selecting previously unselected package zstd.
(Reading database ... 126952 files and directories currently installed.)
Preparing to unpack .../zs

In [ ]:
!curl -fsSL https://ollama.com/install.sh | sh

>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [ ]:
!ollama --version

In [ ]:
import subprocess
import time
import os

# Start Ollama server as a detached process.
# Using a shell command with `nohup`, `&`, and `disown` to ensure it runs truly
# in the background and is not terminated with the Python process.
# Redirect output to a log file for debugging.
ollama_log_file = "/tmp/ollama_server.log"
subprocess.run(
    f"nohup ollama serve > {ollama_log_file} 2>&1 & disown",
    shell=True,
    check=False
)

print(f"Ollama server command sent to background. Check logs at {ollama_log_file} for details.")

# Give the server more time to start up and become ready
time.sleep(15)

# Verify if the server is listening on port 11434
try:
    # Use netstat to check if port 11434 is listening
    output = subprocess.check_output("netstat -tuln | grep 11434", shell=True, text=True)
    if "11434" in output:
        print("Ollama server appears to be listening on port 11434.")
    else:
        print("Ollama server is not listening on port 11434. Checking logs...")
        if os.path.exists(ollama_log_file):
            with open(ollama_log_file, "r") as f:
                print("--- Ollama Server Log ---")
                print(f.read())
                print("-------------------------")
        else:
            print("Ollama server log file not found.")
except subprocess.CalledProcessError:
    print("netstat command failed or port 11434 not found. Checking logs...")
    if os.path.exists(ollama_log_file):
        with open(ollama_log_file, "r") as f:
            print("--- Ollama Server Log ---")
            print(f.read())
            print("-------------------------")
    else:
        print("Ollama server log file not found.")

print("Attempting to connect to Ollama server.")

Ollama server command sent to background. Check logs at /tmp/ollama_server.log for details.
Ollama server appears to be listening on port 11434.
Attempting to connect to Ollama server.


In [ ]:
!ollama pull llama3.2

In [ ]:
!ollama list


NAME               ID              SIZE      MODIFIED       
llama3.2:latest    a80c4f17acd5    2.0 GB    29 seconds ago    


In [ ]:
import requests

response = requests.post(
    "http://127.0.0.1:11434/api/chat",
    json={
        "model": "llama3.2",
        "messages": [
            {
                "role": "user",
                "content": "Hello! Introduce yourself."
            }
        ],
        "stream": False
    }
)

print(response.json()["message"]["content"])

Nice to meet you! I'm an artificial intelligence model, which means I'm a computer program designed to simulate human-like conversations and answer questions to the best of my ability. I don't have a personal name, but I'm here to help and provide information on a wide range of topics.

I'm a large language model, which means I've been trained on a massive dataset of text from the internet, books, and other sources. This training allows me to understand and respond to a variety of questions and topics, from science and history to entertainment and culture.

I'm constantly learning and improving, so please bear with me if I make any mistakes or don't fully understand what you're asking. I'm here to help and provide information, and I'm happy to chat with you about anything that's on your mind!


In [ ]:
!pip install -q streamlit requests pandas openpyxl pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 112.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 33.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 108.0 MB/s eta 0:00:00


In [ ]:
from pathlib import Path

Path("LocalAI-Chat/data").mkdir(parents=True, exist_ok=True)
Path("LocalAI-Chat/uploads").mkdir(parents=True, exist_ok=True)
Path("LocalAI-Chat/.streamlit").mkdir(parents=True, exist_ok=True)

print("Project folders created successfully.")

Project folders created successfully.


In [ ]:
%%writefile LocalAI-Chat/database.py

import sqlite3
import os
from datetime import datetime

DB_DIR = "data"
DB_PATH = os.path.join(DB_DIR, "chat_history.db")


def get_connection():
    os.makedirs(DB_DIR, exist_ok=True)

    conn = sqlite3.connect(
        DB_PATH,
        check_same_thread=False
    )

    conn.row_factory = sqlite3.Row
    return conn


def init_db():
    conn = get_connection()
    cursor = conn.cursor()

    cursor.execute("""
        CREATE TABLE IF NOT EXISTS conversations (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            title TEXT NOT NULL,
            model TEXT,
            created_at TEXT NOT NULL,
            updated_at TEXT NOT NULL
        )
    """)

    cursor.execute("""
        CREATE TABLE IF NOT EXISTS messages (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            conversation_id INTEGER NOT NULL,
            role TEXT NOT NULL,
            content TEXT NOT NULL,
            created_at TEXT NOT NULL,
            FOREIGN KEY(conversation_id)
            REFERENCES conversations(id)
        )
    """)

    conn.commit()
    conn.close()


def create_conversation(
    title="New Chat",
    model="llama3.2"
):
    conn = get_connection()
    cursor = conn.cursor()

    now = datetime.now().isoformat()

    cursor.execute("""
        INSERT INTO conversations
        (title, model, created_at, updated_at)
        VALUES (?, ?, ?, ?)
    """, (title, model, now, now))

    conversation_id = cursor.lastrowid

    conn.commit()
    conn.close()

    return conversation_id


def get_conversations():
    conn = get_connection()
    cursor = conn.cursor()

    cursor.execute("""
        SELECT *
        FROM conversations
        ORDER BY updated_at DESC
    """)

    result = cursor.fetchall()

    conn.close()

    return result


def get_messages(conversation_id):
    conn = get_connection()
    cursor = conn.cursor()

    cursor.execute("""
        SELECT role, content, created_at
        FROM messages
        WHERE conversation_id = ?
        ORDER BY id ASC
    """, (conversation_id,))

    result = cursor.fetchall()

    conn.close()

    return result


def add_message(
    conversation_id,
    role,
    content
):
    conn = get_connection()
    cursor = conn.cursor()

    now = datetime.now().isoformat()

    cursor.execute("""
        INSERT INTO messages
        (conversation_id, role, content, created_at)
        VALUES (?, ?, ?, ?)
    """, (
        conversation_id,
        role,
        content,
        now
    ))

    cursor.execute("""
        UPDATE conversations
        SET updated_at = ?
        WHERE id = ?
    """, (
        now,
        conversation_id
    ))

    conn.commit()
    conn.close()


def update_title(
    conversation_id,
    title
):
    conn = get_connection()
    cursor = conn.cursor()

    cursor.execute("""
        UPDATE conversations
        SET title = ?
        WHERE id = ?
    """, (
        title,
        conversation_id
    ))

    conn.commit()
    conn.close()


def delete_conversation(
    conversation_id
):
    conn = get_connection()
    cursor = conn.cursor()

    cursor.execute("""
        DELETE FROM messages
        WHERE conversation_id = ?
    """, (conversation_id,))

    cursor.execute("""
        DELETE FROM conversations
        WHERE id = ?
    """, (conversation_id,))

    conn.commit()
    conn.close()


def clear_all():
    conn = get_connection()
    cursor = conn.cursor()

    cursor.execute("DELETE FROM messages")
    cursor.execute("DELETE FROM conversations")

    conn.commit()
    conn.close()

Writing LocalAI-Chat/database.py


In [ ]:

%%writefile LocalAI-Chat/ollama_client.py

import requests
import json

OLLAMA_URL = "http://127.0.0.1:11434"


def check_ollama():

    try:
        response = requests.get(
            f"{OLLAMA_URL}/api/tags",
            timeout=5
        )

        return response.status_code == 200

    except Exception:
        return False


def get_models():

    try:
        response = requests.get(
            f"{OLLAMA_URL}/api/tags",
            timeout=5
        )

        data = response.json()

        return [
            model["name"]
            for model in data.get("models", [])
        ]

    except Exception:
        return []


def chat_stream(
    messages,
    model,
    temperature=0.7,
    system_prompt=""
):

    url = f"{OLLAMA_URL}/api/chat"

    final_messages = []

    if system_prompt.strip():

        final_messages.append({
            "role": "system",
            "content": system_prompt
        })

    final_messages.extend(messages)

    payload = {
        "model": model,
        "messages": final_messages,
        "stream": True,
        "options": {
            "temperature": temperature
        }
    }

    try:

        response = requests.post(
            url,
            json=payload,
            stream=True,
            timeout=600
        )

        response.raise_for_status()

        for line in response.iter_lines():

            if not line:
                continue

            try:

                data = json.loads(
                    line.decode("utf-8")
                )

                if "message" in data:

                    content = data["message"].get(
                        "content",
                        ""
                    )

                    if content:
                        yield content

            except Exception:
                continue

    except Exception as error:

        yield f"\n\nOllama Error: {error}"

Writing LocalAI-Chat/ollama_client.py


In [ ]:
import sys
sys.path.append("/content/LocalAI-Chat")

from ollama_client import check_ollama, get_models

print("Ollama running:", check_ollama())
print("Available models:", get_models())

Ollama running: True
Available models: ['llama3.2:latest']


In [ ]:
%%writefile LocalAI-Chat/app.py

import streamlit as st
import pandas as pd
from pypdf import PdfReader
from datetime import datetime

from database import (
    init_db,
    create_conversation,
    get_conversations,
    get_messages,
    add_message,
    update_title,
    delete_conversation,
    clear_all
)

from ollama_client import (
    check_ollama,
    get_models,
    chat_stream
)


# ============================================================
# PAGE CONFIGURATION
# ============================================================

st.set_page_config(
    page_title="LocalAI — Private AI Workspace",
    page_icon="✦",
    layout="wide",
    initial_sidebar_state="expanded"
)


# ============================================================
# DATABASE
# ============================================================

init_db()


# ============================================================
# SESSION STATE
# ============================================================

defaults = {
    "conversation_id": None,
    "system_prompt": (
        "You are a helpful, intelligent, accurate and friendly "
        "AI assistant. Give clear answers and use examples "
        "when useful."
    ),
    "search_query": "",
    "show_settings": False,
    "temperature": 0.7
}

for key, value in defaults.items():

    if key not in st.session_state:
        st.session_state[key] = value


# ============================================================
# PREMIUM CSS
# ============================================================

st.markdown(
    """
<style>

/* ==========================================================
   GLOBAL
   ========================================================== */

#MainMenu {
    visibility: hidden;
}

footer {
    visibility: hidden;
}

header {
    visibility: hidden;
}

.stApp {

    background:
        radial-gradient(
            circle at 10% 0%,
            rgba(124,92,255,0.12),
            transparent 28%
        ),
        radial-gradient(
            circle at 90% 100%,
            rgba(45,212,191,0.08),
            transparent 28%
        ),
        #080A0F;

    color: #F5F7FA;
}


/* ==========================================================
   SIDEBAR
   ========================================================== */

section[data-testid="stSidebar"] {

    background:
        linear-gradient(
            180deg,
            #0B0E14 0%,
            #080A0F 100%
        );

    border-right: 1px solid #202531;
}

section[data-testid="stSidebar"] > div {

    padding-top: 18px;
}


/* ==========================================================
   BRAND
   ========================================================== */

.brand {

    display: flex;
    align-items: center;
    gap: 11px;

    padding: 8px 5px 20px 5px;
}

.brand-icon {

    width: 39px;
    height: 39px;

    display: flex;
    align-items: center;
    justify-content: center;

    border-radius: 12px;

    background:
        linear-gradient(
            135deg,
            #7C5CFF,
            #4F46E5
        );

    box-shadow:
        0 8px 25px rgba(124,92,255,0.30);

    font-size: 21px;
}

.brand-name {

    font-size: 20px;
    font-weight: 800;
    letter-spacing: -0.5px;
}

.brand-sub {

    color: #737C8C;
    font-size: 11px;
}


/* ==========================================================
   BUTTONS
   ========================================================== */

.stButton > button {

    width: 100%;

    border-radius: 11px;

    border: 1px solid #252B38;

    background: #10141C;

    color: #E9ECF1;

    min-height: 42px;

    transition:
        border-color 0.2s,
        background 0.2s,
        transform 0.2s;
}

.stButton > button:hover {

    border-color: #6650D9;

    background: #151927;

    transform: translateY(-1px);
}


/* ==========================================================
   NEW CHAT
   ========================================================== */

.new-chat button {

    background:
        linear-gradient(
            135deg,
            #7C5CFF,
            #5B4BD8
        ) !important;

    border: none !important;

    color: white !important;

    font-weight: 700 !important;

    box-shadow:
        0 8px 25px rgba(124,92,255,0.22);
}


/* ==========================================================
   TOP BAR
   ========================================================== */

.topbar {

    height: 65px;

    display: flex;

    align-items: center;

    justify-content: space-between;

    padding: 0 4px;

    border-bottom: 1px solid #1C212B;

    margin-bottom: 10px;
}

.top-title {

    font-size: 17px;

    font-weight: 700;

    color: #E9ECF1;
}

.top-subtitle {

    font-size: 12px;

    color: #70798A;

    margin-top: 2px;
}

.online-dot {

    display: inline-flex;

    align-items: center;

    gap: 7px;

    padding: 7px 11px;

    border-radius: 20px;

    background: rgba(45,212,191,0.08);

    border: 1px solid rgba(45,212,191,0.18);

    color: #5EEAD4;

    font-size: 12px;

    font-weight: 600;
}


/* ==========================================================
   WELCOME
   ========================================================== */

.welcome {

    text-align: center;

    padding-top: 45px;

    padding-bottom: 30px;
}

.welcome-orb {

    width: 76px;
    height: 76px;

    margin: auto;

    display: flex;

    align-items: center;
    justify-content: center;

    border-radius: 25px;

    background:
        radial-gradient(
            circle at 30% 25%,
            #A78BFA,
            #6D4AFF 40%,
            #29205E
        );

    box-shadow:
        0 20px 60px rgba(124,92,255,0.28);

    font-size: 36px;
}

.welcome h1 {

    margin-top: 22px;

    font-size: 36px;

    font-weight: 800;

    letter-spacing: -1.5px;

    background:
        linear-gradient(
            90deg,
            #FFFFFF,
            #C4B5FD,
            #5EEAD4
        );

    -webkit-background-clip: text;
    -webkit-text-fill-color: transparent;
}

.welcome p {

    max-width: 600px;

    margin: auto;

    color: #7E8797;

    font-size: 15px;

    line-height: 1.7;
}


/* ==========================================================
   FEATURE CARDS
   ========================================================== */

.feature-card {

    min-height: 150px;

    padding: 21px;

    border-radius: 18px;

    background:
        linear-gradient(
            145deg,
            rgba(20,24,34,0.96),
            rgba(12,15,21,0.96)
        );

    border: 1px solid #242A36;

    transition:
        transform 0.2s,
        border-color 0.2s;
}

.feature-card:hover {

    transform: translateY(-3px);

    border-color: #5142A4;
}

.feature-icon {

    font-size: 27px;

    margin-bottom: 10px;
}

.feature-title {

    font-size: 16px;

    font-weight: 700;

    margin-bottom: 7px;
}

.feature-description {

    color: #7E8797;

    font-size: 13px;

    line-height: 1.55;
}


/* ==========================================================
   CHAT
   ========================================================== */

[data-testid="stChatMessage"] {

    border-radius: 17px;

    padding: 12px 16px;

    margin-bottom: 12px;
}

[data-testid="stChatMessage"] p {

    line-height: 1.65;
}


/* ==========================================================
   CHAT INPUT
   ========================================================== */

[data-testid="stChatInput"] {

    border: 1px solid #303746 !important;

    border-radius: 18px !important;

    background: #10141C !important;

    box-shadow:
        0 12px 40px rgba(0,0,0,0.25);
}

[data-testid="stChatInput"] textarea {

    color: #F5F7FA !important;
}


/* ==========================================================
   STATUS CARD
   ========================================================== */

.status-card {

    padding: 12px 14px;

    border-radius: 12px;

    background: #0F141B;

    border: 1px solid #222936;

    font-size: 12px;

    color: #8B95A6;
}


/* ==========================================================
   DIVIDER
   ========================================================== */

.soft-divider {

    height: 1px;

    background: #1D222D;

    margin: 13px 0;
}


/* ==========================================================
   EMPTY SEARCH
   ========================================================== */

.empty-search {

    text-align: center;

    color: #697384;

    padding: 20px;

    font-size: 13px;
}


/* ==========================================================
   FOOTER
   ========================================================== */

.app-footer {

    text-align: center;

    color: #4F5868;

    font-size: 11px;

    padding: 12px 0 3px;
}


/* ==========================================================
   CODE
   ========================================================== */

code {

    border-radius: 5px;
}


/* ==========================================================
   EXPANDER
   ========================================================== */

.streamlit-expanderHeader {

    border-radius: 10px;
}


/* ==========================================================
   MOBILE
   ========================================================== */

@media (max-width: 900px) {

    .welcome h1 {
        font-size: 28px;
    }

    .welcome {
        padding-top: 25px;
    }

}

</style>
""",
    unsafe_allow_html=True
)


# ============================================================
# OLLAMA
# ============================================================

ollama_online = check_ollama()

models = get_models()

if not models:
    models = ["llama3.2:latest"]


# ============================================================
# SIDEBAR
# ============================================================

with st.sidebar:

    # --------------------------------------------------------
    # BRAND
    # --------------------------------------------------------

    st.markdown(
        """
        <div class="brand">

            <div class="brand-icon">
                ✦
            </div>

            <div>
                <div class="brand-name">
                    LocalAI
                </div>

                <div class="brand-sub">
                    PRIVATE AI WORKSPACE
                </div>
            </div>

        </div>
        """,
        unsafe_allow_html=True
    )


    # --------------------------------------------------------
    # NEW CHAT
    # --------------------------------------------------------

    st.markdown(
        '<div class="new-chat">',
        unsafe_allow_html=True
    )

    if st.button(
        "＋  New conversation",
        use_container_width=True
    ):

        st.session_state.conversation_id = (
            create_conversation(
                title="New Chat",
                model=models[0]
            )
        )

        st.rerun()

    st.markdown("</div>", unsafe_allow_html=True)


    # --------------------------------------------------------
    # SEARCH
    # --------------------------------------------------------

    search = st.text_input(
        "Search",
        placeholder="🔍 Search conversations...",
        label_visibility="collapsed"
    )

    st.session_state.search_query = search


    # --------------------------------------------------------
    # CONVERSATIONS
    # --------------------------------------------------------

    st.markdown(
        "##### RECENT CONVERSATIONS"
    )

    conversations = get_conversations()

    if search:

        conversations = [
            c for c in conversations
            if search.lower()
            in c["title"].lower()
        ]


    if conversations:

        for conversation in conversations:

            title = conversation["title"]

            if len(title) > 27:

                title = title[:27] + "..."

            if (
                conversation["id"]
                == st.session_state.conversation_id
            ):

                icon = "🟣"

            else:

                icon = "💬"

            if st.button(
                f"{icon}  {title}",
                key=f"chat_{conversation['id']}",
                use_container_width=True
            ):

                st.session_state.conversation_id = (
                    conversation["id"]
                )

                st.rerun()

    else:

        st.markdown(
            """
            <div class="empty-search">
                No conversations found.
            </div>
            """,
            unsafe_allow_html=True
        )


    st.markdown(
        '<div class="soft-divider"></div>',
        unsafe_allow_html=True
    )


    # --------------------------------------------------------
    # MODEL
    # --------------------------------------------------------

    st.markdown("##### 🧠 AI MODEL")

    selected_model = st.selectbox(
        "Model",
        models,
        label_visibility="collapsed"
    )


    # --------------------------------------------------------
    # SETTINGS
    # --------------------------------------------------------

    with st.expander(
        "⚙️  AI Settings"
    ):

        st.session_state.temperature = st.slider(
            "Temperature",
            0.0,
            1.5,
            st.session_state.temperature,
            0.1,
            help=(
                "Lower values make responses more focused. "
                "Higher values make responses more creative."
            )
        )

        st.session_state.system_prompt = st.text_area(
            "System instructions",
            value=st.session_state.system_prompt,
            height=150
        )


    # --------------------------------------------------------
    # FILE UPLOAD
    # --------------------------------------------------------

    with st.expander(
        "📎  Documents"
    ):

        uploaded_file = st.file_uploader(
            "Upload a document",
            type=[
                "pdf",
                "txt",
                "csv",
                "xlsx"
            ],
            help="Ask questions about your uploaded file."
        )

        if uploaded_file:

            st.success(
                f"✓ {uploaded_file.name}"
            )

            st.caption(
                "Supported: PDF • TXT • CSV • XLSX"
            )

        else:

            st.caption(
                "Upload a document to analyze it with AI."
            )


    # --------------------------------------------------------
    # STATUS
    # --------------------------------------------------------

    if ollama_online:

        st.markdown(
            """
            <div class="status-card">
                🟢 <b>Ollama Online</b><br>
                <span style="color:#697384">
                Local inference active
                </span>
            </div>
            """,
            unsafe_allow_html=True
        )

    else:

        st.markdown(
            """
            <div class="status-card">
                🔴 <b>Ollama Offline</b><br>
                <span style="color:#697384">
                Start Ollama server
                </span>
            </div>
            """,
            unsafe_allow_html=True
        )


    st.markdown(
        '<div class="soft-divider"></div>',
        unsafe_allow_html=True
    )


    # --------------------------------------------------------
    # CHAT MANAGEMENT
    # --------------------------------------------------------

    if st.session_state.conversation_id:

        with st.expander(
            "🛠 Chat management"
        ):

            current_chat = None

            for c in conversations:

                if (
                    c["id"]
                    == st.session_state.conversation_id
                ):

                    current_chat = c
                    break


            # Rename

            if current_chat:

                new_title = st.text_input(
                    "Rename chat",
                    value=current_chat["title"],
                    key="rename_chat"
                )

                if st.button(
                    "Save name",
                    use_container_width=True
                ):

                    if new_title.strip():

                        update_title(
                            st.session_state.conversation_id,
                            new_title.strip()
                        )

                        st.rerun()


            # Delete current

            if st.button(
                "🗑 Delete current chat",
                use_container_width=True
            ):

                delete_conversation(
                    st.session_state.conversation_id
                )

                st.session_state.conversation_id = None

                st.rerun()


    # --------------------------------------------------------
    # CLEAR ALL
    # --------------------------------------------------------

    if st.button(
        "🗑️ Clear all conversations",
        use_container_width=True
    ):

        clear_all()

        st.session_state.conversation_id = None

        st.rerun()


    # --------------------------------------------------------
    # SIDEBAR FOOTER
    # --------------------------------------------------------

    st.markdown(
        """
        <div class="app-footer">

            ✦ LocalAI<br>

            Your conversations stay in your
            local database.

        </div>
        """,
        unsafe_allow_html=True
    )


# ============================================================
# MAIN HEADER
# ============================================================

col_left, col_right = st.columns(
    [8, 2]
)

with col_left:

    st.markdown(
        """
        <div class="topbar">

            <div>
                <div class="top-title">
                    LocalAI Assistant
                </div>

                <div class="top-subtitle">
                    Private conversational AI
                </div>
            </div>

        </div>
        """,
        unsafe_allow_html=True
    )


with col_right:

    if ollama_online:

        st.markdown(
            """
            <div style="text-align:right; padding-top:12px;">
                <span class="online-dot">
                    ● Online
                </span>
            </div>
            """,
            unsafe_allow_html=True
        )

    else:

        st.markdown(
            """
            <div style="text-align:right; padding-top:12px;">
                <span style="
                    color:#F87171;
                    font-size:12px;
                ">
                    ● Offline
                </span>
            </div>
            """,
            unsafe_allow_html=True
        )


# ============================================================
# OLLAMA ERROR
# ============================================================

if not ollama_online:

    st.error(
        """
        Ollama is currently offline.

        Start it with:

        `ollama serve`
        """
    )

    st.stop()


# ============================================================
# CREATE INITIAL CHAT
# ============================================================

if st.session_state.conversation_id is None:

    st.session_state.conversation_id = (
        create_conversation(
            title="New Chat",
            model=selected_model
        )
    )

    st.rerun()


# ============================================================
# CURRENT MESSAGES
# ============================================================

messages_db = get_messages(
    st.session_state.conversation_id
)


# ============================================================
# WELCOME SCREEN
# ============================================================

if not messages_db:

    st.markdown(
        """
        <div class="welcome">

            <div class="welcome-orb">
                ✦
            </div>

            <h1>
                What can I help you with?
            </h1>

            <p>
                Ask questions, write and debug code,
                analyze data, work with documents,
                learn concepts, or brainstorm ideas.
            </p>

        </div>
        """,
        unsafe_allow_html=True
    )


    col1, col2, col3 = st.columns(3)


    with col1:

        st.markdown(
            """
            <div class="feature-card">

                <div class="feature-icon">
                    💻
                </div>

                <div class="feature-title">
                    Code & Debug
                </div>

                <div class="feature-description">
                    Python, SQL, Java, debugging,
                    algorithms and project help.
                </div>

            </div>
            """,
            unsafe_allow_html=True
        )


    with col2:

        st.markdown(
            """
            <div class="feature-card">

                <div class="feature-icon">
                    📊
                </div>

                <div class="feature-title">
                    Data Analysis
                </div>

                <div class="feature-description">
                    Analyze CSV and Excel data,
                    SQL queries and analytics.
                </div>

            </div>
            """,
            unsafe_allow_html=True
        )


    with col3:

        st.markdown(
            """
            <div class="feature-card">

                <div class="feature-icon">
                    📚
                </div>

                <div class="feature-title">
                    Learn Anything
                </div>

                <div class="feature-description">
                    Get simple explanations,
                    examples and study guidance.
                </div>

            </div>
            """,
            unsafe_allow_html=True
        )


    st.markdown(
        "<br>",
        unsafe_allow_html=True
    )


    st.caption(
        "Try one of these prompts:"
    )


    prompt_col1, prompt_col2 = st.columns(2)


    with prompt_col1:

        st.info(
            "💡 Explain SQL JOINs with a simple example."
        )

        st.info(
            "💡 Help me create a Data Analyst resume."
        )


    with prompt_col2:

        st.info(
            "💡 Write a Python data analysis project."
        )

        st.info(
            "💡 Explain machine learning step by step."
        )


# ============================================================
# DISPLAY CHAT HISTORY
# ============================================================

for message in messages_db:

    role = message["role"]

    content = message["content"]


    if role not in ["user", "assistant"]:
        continue


    with st.chat_message(
        "user" if role == "user" else "assistant"
    ):

        st.markdown(content)


# ============================================================
# FILE TEXT EXTRACTION
# ============================================================

def extract_file_text(file):

    if file is None:

        return ""


    filename = file.name.lower()


    # --------------------------------------------------------
    # TXT
    # --------------------------------------------------------

    if filename.endswith(".txt"):

        return file.read().decode(
            "utf-8",
            errors="ignore"
        )


    # --------------------------------------------------------
    # CSV
    # --------------------------------------------------------

    if filename.endswith(".csv"):

        df = pd.read_csv(file)

        return df.head(200).to_string(
            index=False
        )


    # --------------------------------------------------------
    # XLSX
    # --------------------------------------------------------

    if filename.endswith(".xlsx"):

        df = pd.read_excel(file)

        return df.head(200).to_string(
            index=False
        )


    # --------------------------------------------------------
    # PDF
    # --------------------------------------------------------

    if filename.endswith(".pdf"):

        reader = PdfReader(file)

        pages = []

        for page in reader.pages:

            text = page.extract_text()

            if text:

                pages.append(text)


        return "\n\n".join(
            pages
        )[:40000]


    return ""


# ============================================================
# CHAT INPUT
# ============================================================

prompt = st.chat_input(
    "Message LocalAI..."
)


# ============================================================
# PROCESS PROMPT
# ============================================================

if prompt:

    # --------------------------------------------------------
    # FILE CONTEXT
    # --------------------------------------------------------

    file_context = extract_file_text(
        uploaded_file
    )


    final_prompt = prompt


    if file_context:

        final_prompt = f"""
You are analyzing a user-provided document.

Use the document information below when answering.

DOCUMENT:
==================================================
{file_context}
==================================================

USER QUESTION:
{prompt}

Instructions:
- Answer based on the document when relevant.
- Clearly explain your reasoning.
- If the answer cannot be found in the document,
  say so instead of inventing information.
"""


    # --------------------------------------------------------
    # SAVE USER MESSAGE
    # --------------------------------------------------------

    add_message(
        st.session_state.conversation_id,
        "user",
        final_prompt
    )


    # --------------------------------------------------------
    # AUTO TITLE
    # --------------------------------------------------------

    current_messages = get_messages(
        st.session_state.conversation_id
    )


    if len(current_messages) == 1:

        clean_title = prompt.strip()

        if len(clean_title) > 45:

            clean_title = clean_title[:45] + "..."

        update_title(
            st.session_state.conversation_id,
            clean_title
        )


    # --------------------------------------------------------
    # SHOW USER MESSAGE
    # --------------------------------------------------------

    with st.chat_message("user"):

        st.markdown(prompt)


    # --------------------------------------------------------
    # PREPARE HISTORY
    # --------------------------------------------------------

    history = get_messages(
        st.session_state.conversation_id
    )


    ollama_messages = []


    for message in history:

        ollama_messages.append(
            {
                "role": message["role"],
                "content": message["content"]
            }
        )


    # --------------------------------------------------------
    # AI RESPONSE
    # --------------------------------------------------------

    with st.chat_message("assistant"):

        response_placeholder = st.empty()

        full_response = ""


        for chunk in chat_stream(
            ollama_messages,
            selected_model,
            st.session_state.temperature,
            st.session_state.system_prompt
        ):

            full_response += chunk

            response_placeholder.markdown(
                full_response + "▌"
            )


        response_placeholder.markdown(
            full_response
        )


    # --------------------------------------------------------
    # SAVE AI RESPONSE
    # --------------------------------------------------------

    add_message(
        st.session_state.conversation_id,
        "assistant",
        full_response
    )


    # --------------------------------------------------------
    # RERUN
    # --------------------------------------------------------

    st.rerun()

Writing LocalAI-Chat/app.py


In [ ]:
!pkill -f "streamlit run" || true

^C


In [ ]:
!cd LocalAI-Chat && nohup streamlit run app.py --server.address 0.0.0.0 --server.port 8501 > /tmp/streamlit.log 2>&1 &

In [ ]:
import time
time.sleep(8)

!cat /tmp/streamlit.log



2026-09-16 08:03:43.757 Uvicorn server started on 0.0.0.0:8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.158.52.59:8501

  Stopping...


In [ ]:
import subprocess
import time
import os

# Stop old Streamlit processes
subprocess.run(
    ["pkill", "-f", "streamlit"],
    capture_output=True
)

time.sleep(2)

# Start Streamlit properly in background
log_file = open("/tmp/streamlit.log", "w")

streamlit_process = subprocess.Popen(
    [
        "streamlit",
        "run",
        "/content/LocalAI-Chat/app.py",
        "--server.address=0.0.0.0",
        "--server.port=8501",
        "--server.headless=true"
    ],
    stdout=log_file,
    stderr=subprocess.STDOUT,
    start_new_session=True
)

time.sleep(8)

print("Streamlit process started.")
print("PID:", streamlit_process.pid)

Streamlit process started.
PID: 7195


In [ ]:
print("========== STREAMLIT LOG ==========")

with open("/tmp/streamlit.log", "r") as f:
    print(f.read())

========== STREAMLIT LOG ==========


2026-09-16 08:10:18.160 Uvicorn server started on 0.0.0.0:8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.158.52.59:8501




In [ ]:
import socket

sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)

result = sock.connect_ex(("127.0.0.1", 8501))

sock.close()

if result == 0:
    print("✅ Streamlit is running on port 8501")
else:
    print("❌ Streamlit is NOT running")

✅ Streamlit is running on port 8501


In [ ]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /content/cloudflared
!chmod +x /content/cloudflared

print("Cloudflared installed.")

Cloudflared installed.


In [ ]:
import subprocess
import time
import re

cloudflared_process = subprocess.Popen(
    [
        "/content/cloudflared",
        "tunnel",
        "--url",
        "http://127.0.0.1:8501"
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1
)

public_url = None

start_time = time.time()

while time.time() - start_time < 30:

    line = cloudflared_process.stdout.readline()

    if line:

        print(line.strip())

        match = re.search(
            r"https://[a-zA-Z0-9-]+\.trycloudflare\.com",
            line
        )

        if match:

            public_url = match.group(0)
            break


if public_url:

    print("\n" + "=" * 60)
    print("🚀 YOUR LOCALAI APP IS LIVE")
    print("=" * 60)
    print(public_url)
    print("=" * 60)

else:

    print("❌ Public URL was not detected.")
    print("Check the Cloudflare output above.")

2026-09-16T08:11:42Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-09-16T08:11:42Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-09-16T08:11:47Z INF +--------------------------------------------------------------------------------------------+
2026-09-16T08:11:47Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2026-09-16T08:11:47Z INF |  https://transaction-recordings-beast-hollywood.tryclo

In [ ]:
import socket

sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
result = sock.connect_ex(("127.0.0.1", 8501))
sock.close()

if result == 0:
    print("✅ Streamlit is running")
else:
    print("❌ Streamlit is not running")

✅ Streamlit is running


In [ ]:
import subprocess
import time
import re

# Stop old cloudflared process
subprocess.run(
    ["pkill", "-f", "cloudflared"],
    capture_output=True
)

time.sleep(2)

# Start a fresh tunnel
cloudflared_process = subprocess.Popen(
    [
        "/content/cloudflared",
        "tunnel",
        "--url",
        "http://127.0.0.1:8501"
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1
)

print("Starting new Cloudflare tunnel...\n")

public_url = None
start_time = time.time()

while time.time() - start_time < 30:

    line = cloudflared_process.stdout.readline()

    if line:
        print(line.strip())

        match = re.search(
            r"https://[a-zA-Z0-9-]+\.trycloudflare\.com",
            line
        )

        if match:
            public_url = match.group(0)
            break

if public_url:
    print("\n" + "=" * 60)
    print("🚀 NEW LOCALAI PUBLIC URL")
    print("=" * 60)
    print(public_url)
    print("=" * 60)
else:
    print("\n❌ Tunnel URL was not detected.")

Starting new Cloudflare tunnel...

2026-09-16T08:13:20Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-09-16T08:13:20Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-09-16T08:13:26Z INF +--------------------------------------------------------------------------------------------+
2026-09-16T08:13:26Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2026-09-16T08:13:26Z INF |  https://camcorders

In [ ]:
!curl http://localhost:8080/v1/models


In [ ]:
!cloudflared tunnel --url http://localhost:8080

/bin/bash: line 1: cloudflared: command not found


In [ ]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb

Selecting previously unselected package cloudflared.
(Reading database ... 126973 files and directories currently installed.)
Preparing to unpack cloudflared-linux-amd64.deb ...
Unpacking cloudflared (2026.9.1) ...
Setting up cloudflared (2026.9.1) ...
Processing triggers for man-db (2.12.0-4build2) ...


In [ ]:
!cloudflared --version

cloudflared version 2026.9.1 (built 2026-09-11-13:35 UTC)


In [ ]:
!curl http://localhost:8080/v1/models

In [ ]:
!cloudflared tunnel --url http://localhost:8080

2026-09-16T08:23:58Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-09-16T08:23:58Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-09-16T08:24:04Z INF +--------------------------------------------------------------------------------------------+
2026-09-16T08:24:04Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2026-09-16T08:24:04Z INF |  https://render-hostel-publisher-plant.trycloudflare.c

In [ ]:
!curl http://localhost:8080/v1/models

In [ ]:
!streamlit run app.py --server.address 0.0.0.0 --server.port 8501 > streamlit.log 2>&1 &

In [ ]:
!curl -I http://localhost:8501

HTTP/1.1 200 OK
date: Wed, 16 Sep 2026 08:26:55 GMT
server: uvicorn
content-type: text/html; charset=utf-8
accept-ranges: bytes
content-length: 7260
last-modified: Wed, 16 Sep 2026 07:58:48 GMT
etag: "fe8a15413785cb15bc113b98873e9e21"
cache-control: no-cache



In [ ]:
!cloudflared tunnel --url http://localhost:8501

2026-09-16T08:27:22Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-09-16T08:27:22Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-09-16T08:27:27Z INF +--------------------------------------------------------------------------------------------+
2026-09-16T08:27:27Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2026-09-16T08:27:27Z INF |  https://markers-latest-nil-fifteen.trycloudflare.com 

In [ ]:
!curl http://localhost:8080/v1/models